# GIAI ĐOẠN 1 – APRIORI (bản sửa)
**Đề tài:** Phân tích giỏ hàng siêu thị và xây dựng gợi ý mua kèm

Notebook này chạy Apriori tự cài đặt, đối chiếu với `mlxtend`, khảo sát lưới 5 support × 4 confidence và xuất danh sách luật.

**Những điểm đã sửa so với bản cũ:**
1. Dùng đúng file đầu vào của báo cáo (`customers_clean_apriori.csv`, 38.006 dòng) và in mã SHA-256 của danh sách giỏ để đối chiếu với cả nhóm.
2. Đo thời gian công bằng: cả hai cách đều chạy 1 lần khởi động rồi lấy trung vị của 3 lần chạy.
3. Sinh luật bằng phép so sánh confidence chính xác (`Fraction`), không phụ thuộc phiên bản `mlxtend` và không lệch 1 luật ở biên.
4. Đối chiếu cả số đếm và support của từng tập phổ biến (không chỉ tên tập), kèm kiểm tra đếm trực tiếp độc lập.
5. Bổ sung bảng số luật có lift > 1, bước lọc luật thừa và cột số giỏ để đánh giá độ tin cậy của luật.

## 1. Cấu hình và thư viện

In [ ]:
import hashlib, json, subprocess, sys, time, warnings
from collections import Counter
from decimal import Decimal, ROUND_CEILING
from fractions import Fraction
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import mlxtend  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mlxtend"])
from mlxtend.frequent_patterns import apriori as mlx_apriori
from mlxtend.preprocessing import TransactionEncoder

# ===== CẤU HÌNH =====
DATA_CANDIDATES = [Path("customers_clean_apriori.csv"),
                   Path("sample_data/customers_clean_apriori.csv"),
                   Path("data/processed/customers_clean_apriori.csv")]
OUT_DIR = Path("results")
SUPPORTS = [0.001, 0.003, 0.005, 0.0075, 0.010]
CONFS = [0.08, 0.10, 0.12, 0.15]
SO_LAN = 3                       # số lần đo thời gian (lấy trung vị)
S_CHON, C_CHON = 0.001, 0.08     # cấu hình dùng để xuất luật và demo
MIN_GIO_TIN_CAY = 30             # luật có ít hơn số giỏ này chỉ mang tính tham khảo
HASH_THAM_CHIEU = "7910a17405595931e73ac896e71ba1256b1d4b2596f6f5350e7535ce77c99137"

## 2. Đọc dữ liệu và tạo giỏ hàng

In [ ]:
def tim_file():
    for p in DATA_CANDIDATES:
        if p.exists():
            return p
    try:
        from google.colab import files
        up = files.upload()
        return Path(next(iter(up)))
    except ImportError:
        raise FileNotFoundError("Không thấy customers_clean_apriori.csv. Đặt file cạnh notebook "
                                "hoặc sửa DATA_CANDIDATES.")


def doc_du_lieu(path):
    df = pd.read_csv(path)
    thieu = {"Member_number", "Date", "itemDescription"} - set(df.columns)
    if thieu:
        raise ValueError(f"Thiếu cột: {sorted(thieu)}. Các cột hiện có: {list(df.columns)}")
    d = df["Date"].astype(str).str.strip()
    la_iso = d.str.fullmatch(r"\d{4}-\d{2}-\d{2}")
    ngay = pd.to_datetime(d.where(la_iso), format="%Y-%m-%d", errors="coerce")
    ngay = ngay.fillna(pd.to_datetime(d.where(~la_iso), format="%d-%m-%Y", errors="coerce"))
    if ngay.isna().any():
        raise ValueError(f"Có {int(ngay.isna().sum())} ngày không đọc được.")
    df["Date"] = ngay
    df["itemDescription"] = df["itemDescription"].astype(str).str.strip().str.lower()
    return df


def tao_giao_dich(df):
    grouped = df.groupby(["Member_number", "Date"], sort=True)["itemDescription"].agg(list)
    return [tuple(sorted(set(b))) for b in grouped]


file_dl = tim_file()
df = doc_du_lieu(file_dl)
transactions = tao_giao_dich(df)
N = len(transactions)
items = sorted(set().union(*map(set, transactions)))
sha = hashlib.sha256(json.dumps(transactions, ensure_ascii=False,
                                separators=(",", ":")).encode()).hexdigest()

print(f"File: {file_dl} | {len(df):,} dòng")
print(f"Giỏ hàng: {N:,} | Mặt hàng: {len(items)} | "
      f"TB mặt hàng/giỏ: {np.mean([len(t) for t in transactions]):.2f} | "
      f"Lớn nhất: {max(len(t) for t in transactions)}")
print("SHA-256 giỏ hàng:", sha)
print("Khớp mã tham chiếu của nhóm:", sha == HASH_THAM_CHIEU)
if len(df) != 38006:
    print("CẢNH BÁO: báo cáo ghi 38.006 dòng, file này có", len(df), "dòng.")

## 3. Apriori tự cài đặt và sinh luật

In [ ]:
def min_count(support, n):
    """Số giỏ tối thiểu = ceil(support * n), tính bằng Decimal để tránh sai số số thực."""
    return int((Decimal(str(support)) * n).to_integral_value(rounding=ROUND_CEILING))


def apriori_tu_cai_dat(transactions, min_support):
    """Trả về {frozenset(tập mặt hàng): số giỏ chứa tập đó}."""
    n = len(transactions)
    nguong = min_count(min_support, n)
    dem = Counter(item for t in transactions for item in t)
    L = {frozenset([i]): c for i, c in dem.items() if c >= nguong}
    ket_qua = dict(L)
    k = 2
    while L:
        truoc = set(L)
        ung_vien = {}
        for a in truoc:
            for b in truoc:
                u = a | b
                if len(u) == k and u not in ung_vien:
                    # cắt tỉa: mọi tập con k-1 phần tử phải phổ biến
                    if all(frozenset(s) in truoc for s in combinations(u, k - 1)):
                        ung_vien[u] = 0
        if not ung_vien:
            break
        for t in transactions:
            if len(t) >= k:
                for c in combinations(t, k):
                    fs = frozenset(c)
                    if fs in ung_vien:
                        ung_vien[fs] += 1
        L = {s: c for s, c in ung_vien.items() if c >= nguong}
        ket_qua.update(L)
        k += 1
    return ket_qua


COT_LUAT = ["antecedents", "consequents", "so_gio", "support", "confidence", "lift"]


def sinh_luat(freq, min_conf, n):
    """Sinh luật X -> Y từ các tập phổ biến; so sánh confidence bằng phân số chính xác."""
    nguong = Fraction(str(min_conf))
    rows = []
    for itemset, c in freq.items():
        if len(itemset) < 2:
            continue
        for k in range(1, len(itemset)):
            for trai in combinations(sorted(itemset), k):
                trai = frozenset(trai)
                phai = itemset - trai
                c_trai, c_phai = freq[trai], freq[phai]
                if Fraction(c, c_trai) >= nguong:
                    rows.append({"antecedents": trai, "consequents": phai, "so_gio": c,
                                 "support": c / n, "confidence": c / c_trai,
                                 "lift": c * n / (c_trai * c_phai)})
    return pd.DataFrame(rows, columns=COT_LUAT)


def loc_luat_thua(rules):
    """Luật thừa: tồn tại luật tổng quát hơn (vế trái là tập con thực sự, cùng vế phải)
    có confidence không thấp hơn. Trả về bản sao không còn luật thừa."""
    if rules.empty:
        return rules.copy()
    theo_ve_phai = {}
    for r in rules.itertuples():
        theo_ve_phai.setdefault(r.consequents, []).append((r.antecedents, r.confidence))
    giu = [not any(a < r.antecedents and conf >= r.confidence
                   for a, conf in theo_ve_phai[r.consequents])
           for r in rules.itertuples()]
    return rules[giu].copy()

## 4. Kiểm tra tính đúng đắn
Đối chiếu bản tự cài đặt với `mlxtend` (cả tập, số đếm và support), và với phép đếm trực tiếp độc lập.

In [ ]:
te = TransactionEncoder()
df_enc = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
print("Ma trận mã hóa:", df_enc.shape)


def doi_chieu_mlxtend(freq, fi):
    mlx = {frozenset(s): sup for s, sup in zip(fi["itemsets"], fi["support"])}
    if set(mlx) != set(freq):
        return False
    return all(np.isclose(sup, freq[s] / N) for s, sup in mlx.items())


freq_theo_support = {s: apriori_tu_cai_dat(transactions, s) for s in SUPPORTS}

# 1) Đếm trực tiếp mọi tổ hợp xuất hiện, độc lập với Apriori
max_k = max(len(s) for f in freq_theo_support.values() for s in f)
quan_sat = Counter()
for t in transactions:
    for k in range(1, min(len(t), max_k) + 1):
        quan_sat.update(frozenset(c) for c in combinations(t, k))
for s, f in freq_theo_support.items():
    mong_doi = {x: c for x, c in quan_sat.items() if c >= min_count(s, N)}
    assert f == mong_doi, f"Apriori tự cài đặt lệch phép đếm trực tiếp ở support={s}"

# 2) Tính bao hàm giữa các ngưỡng
ds = sorted(SUPPORTS)
for a, b in zip(ds, ds[1:]):
    assert set(freq_theo_support[b]) <= set(freq_theo_support[a])
print("PASS: số đếm của Apriori tự cài đặt khớp phép đếm trực tiếp ở mọi ngưỡng.")

## 5. Thực nghiệm lưới support × confidence
Thời gian đo cho bước khai thác tập phổ biến: một lần khởi động, sau đó lấy **trung vị của 3 lần chạy**, áp dụng giống nhau cho bản tự cài đặt và `mlxtend`.

In [ ]:
def do_thoi_gian(ham, *args, lan=SO_LAN):
    ham(*args)                                  # lần khởi động, không tính
    ts = []
    for _ in range(lan):
        t0 = time.perf_counter()
        ham(*args)
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))


def chay_mlxtend(s):
    return mlx_apriori(df_enc, min_support=s, use_colnames=True)


rows = []
for s in SUPPORTS:
    freq = freq_theo_support[s]
    t_tu_cai = do_thoi_gian(apriori_tu_cai_dat, transactions, s)
    t_mlx = do_thoi_gian(chay_mlxtend, s)
    khop = doi_chieu_mlxtend(freq, chay_mlxtend(s))
    for c in CONFS:
        luat = sinh_luat(freq, c, N)
        rows.append({"min_support": s, "min_conf": c,
                     "so_gio_toi_thieu": min_count(s, N),
                     "so_tap_pho_bien": len(freq),
                     "so_luat": len(luat),
                     "so_luat_lift_gt_1": int((luat["lift"] > 1).sum()),
                     "tg_tu_cai_dat_s": round(t_tu_cai, 4),
                     "tg_mlxtend_s": round(t_mlx, 4),
                     "khop_mlxtend": khop})
bang = pd.DataFrame(rows)
print(bang.to_string(index=False))
assert bang["khop_mlxtend"].all(), "Có ngưỡng mà bản tự cài đặt không khớp mlxtend"
print("\nPASS: tập phổ biến và support khớp mlxtend ở mọi ngưỡng.")

Hai bảng gọn để đưa vào báo cáo: số luật đạt ngưỡng, và số luật có lift > 1 (luật thật sự có quan hệ mua kèm).

In [ ]:
print("Số luật đạt support và confidence:")
display(bang.pivot(index="min_support", columns="min_conf", values="so_luat"))
print("Trong đó, số luật có lift > 1:")
display(bang.pivot(index="min_support", columns="min_conf", values="so_luat_lift_gt_1"))

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
bang.to_csv(OUT_DIR / "apriori_hieu_nang.csv", index=False)

g = bang.drop_duplicates("min_support").sort_values("min_support")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(g["min_support"], g["tg_mlxtend_s"], "o-", label="mlxtend")
ax[0].plot(g["min_support"], g["tg_tu_cai_dat_s"], "s--", label="tự cài đặt")
ax[0].set(xscale="log", xlabel="min_support", ylabel="Thời gian trung vị (s)",
          title="Thời gian chạy Apriori")
ax[0].legend()
ax[1].plot(g["min_support"], g["so_tap_pho_bien"], "o-", color="tab:green")
ax[1].set(xscale="log", xlabel="min_support", ylabel="Số tập phổ biến",
          title="Số tập phổ biến theo support")
for a in ax:
    a.invert_xaxis()
    a.grid(alpha=.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "apriori_hieu_nang.png", dpi=150)
plt.show()

## 6. Luật của cấu hình được chọn
Quy trình: sinh luật → giữ luật có lift > 1 → loại luật thừa. Cột `so_gio` là số giỏ chứa cả hai vế; luật có `so_gio` nhỏ chỉ nên xem là gợi ý tham khảo.

In [ ]:
tat_ca = sinh_luat(freq_theo_support[S_CHON], C_CHON, N)
lift_duong = tat_ca[tat_ca["lift"] > 1]
sach = loc_luat_thua(tat_ca)
sach = sach[sach["lift"] > 1].copy()
sach["du_tin_cay"] = sach["so_gio"] >= MIN_GIO_TIN_CAY

print(f"Cấu hình: support={S_CHON}, confidence={C_CHON}")
print(f"  Luật đạt ngưỡng:                  {len(tat_ca)}")
print(f"  Luật có lift > 1:                 {len(lift_duong)}")
print(f"  Sau khi loại luật thừa:           {len(sach)}")
print(f"  Trong đó có so_gio >= {MIN_GIO_TIN_CAY}:           {int(sach['du_tin_cay'].sum())}")

cach = lambda x: ", ".join(sorted(x))
xuat = sach.assign(antecedents=sach["antecedents"].map(cach),
                   consequents=sach["consequents"].map(cach))
xuat = xuat.sort_values(["lift", "confidence"], ascending=False)
xuat.to_csv(OUT_DIR / "apriori_luat.csv", index=False)
display(xuat.head(15).round(4))

## 7. Gợi ý mua kèm từ luật (dùng cho demo)

In [ ]:
def goi_y(san_pham, luat, top_n=5):
    """Trả về các sản phẩm nên mua kèm khi khách chọn `san_pham`, xếp theo lift rồi confidence."""
    sp = san_pham.strip().lower()
    khop = luat[luat["antecedents"].map(lambda a: a == frozenset([sp]))]
    if khop.empty:
        return f"Chưa có luật mua kèm cho '{sp}' ở cấu hình hiện tại."
    khop = khop.sort_values(["lift", "confidence"], ascending=False)
    khop = khop.assign(goi_y=khop["consequents"].map(cach))
    return khop[["goi_y", "so_gio", "confidence", "lift"]].head(top_n).round(4)


mau = sorted({next(iter(a)) for a in sach["antecedents"] if len(a) == 1})[:3]
for sp in mau:
    print(f"Khách chọn: {sp}")
    display(goi_y(sp, sach))

## 8. Tải kết quả (chỉ chạy trên Colab)

In [ ]:
try:
    from google.colab import files
    for f in ["apriori_hieu_nang.csv", "apriori_luat.csv", "apriori_hieu_nang.png"]:
        files.download(str(OUT_DIR / f))
except ImportError:
    print("Không chạy trên Colab: file nằm trong thư mục", OUT_DIR.resolve())